In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
#load dataset
DATASET_PATH='/home/kaviya/Downloads/archive'
image_paths = []
labels = []
splits = []

for split in ["train", "test"]:
    split_path = os.path.join(DATASET_PATH, split)
    
    for label in ["REAL", "FAKE"]:
        label_path = os.path.join(split_path, label)
        
        for img in os.listdir(label_path):
            image_paths.append(os.path.join(label_path, img))
            labels.append(label)
            splits.append(split)

df = pd.DataFrame({
    "image_path": image_paths,
    "label": labels,
    "split": splits
})

print(df["label"].value_counts())
print(df["split"].value_counts())
df.head()

label
REAL    60000
FAKE    60000
Name: count, dtype: int64
split
train    100000
test      20000
Name: count, dtype: int64


,image_path,label,split
0,/home/kaviya/Downloads/archive/train/REAL/3195...,REAL,train
1,/home/kaviya/Downloads/archive/train/REAL/2745...,REAL,train
2,/home/kaviya/Downloads/archive/train/REAL/4114...,REAL,train
3,/home/kaviya/Downloads/archive/train/REAL/2029...,REAL,train
4,/home/kaviya/Downloads/archive/train/REAL/1659...,REAL,train


In [3]:
#take subset of train set
df_train = df[df["split"] == "train"]

df_train_subset = df_train.groupby("label").apply(
    lambda x: x.sample(2500, random_state=42)
).reset_index(drop=True)

print("Train subset distribution:")
print(df_train_subset["label"].value_counts())

Train subset distribution:
label
FAKE    2500
REAL    2500
Name: count, dtype: int64


/tmp/ipykernel_4404/2794083.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train_subset = df_train.groupby("label").apply(


In [4]:
#take subset of test
df_test = df[df["split"] == "test"]

df_test_subset = df_test.groupby("label").apply(
    lambda x: x.sample(1000, random_state=42)
).reset_index(drop=True)

print("Test subset distribution:")
print(df_test_subset["label"].value_counts())

Test subset distribution:
label
FAKE    1000
REAL    1000
Name: count, dtype: int64


/tmp/ipykernel_4404/3457897849.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test_subset = df_test.groupby("label").apply(


In [7]:
pip install torch torchvision

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [9]:

from torchvision import transforms

IMAGE_SIZE = 224

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [18]:
from torch.utils.data import Dataset
from PIL import Image

class CIFAKEDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
        self.label_map = {"REAL": 0, "FAKE": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["image_path"]
        label = self.label_map[self.df.iloc[idx]["label"]]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [19]:
#create dataset objects
train_dataset = CIFAKEDataset(df_train_subset, transform=train_transform)
test_dataset = CIFAKEDataset(df_test_subset, transform=test_transform)

In [20]:
#create dataloaders
from torch.utils.data import DataLoader

BATCH_SIZE = 32 if torch.cuda.is_available() else 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Batch size:", BATCH_SIZE)

Batch size: 16


In [21]:
#setup resnet
import torchvision.models as models
import torch.nn as nn

model = models.resnet18(pretrained=True)

# If GPU available → train full model
# If CPU → freeze backbone for speed
if torch.cuda.is_available():
    print("Training full model (GPU mode)")
else:
    print("Freezing backbone (CPU mode)")
    for param in model.parameters():
        param.requires_grad = False

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

Freezing backbone (CPU mode)


In [22]:
criterion = nn.CrossEntropyLoss()

if torch.cuda.is_available():
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
else:
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

In [23]:
#training
EPOCHS = 5 if torch.cuda.is_available() else 3

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Train Accuracy: {train_acc:.4f}")

Epoch [1/3] Loss: 0.4924 Train Accuracy: 0.7616
Epoch [2/3] Loss: 0.4253 Train Accuracy: 0.8014
Epoch [3/3] Loss: 0.3868 Train Accuracy: 0.8264
